In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("quote", "\"")
    .option("escape", "\"")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO)
)



In [0]:
for col in bruto.columns:
    print(col)

In [0]:
renomeacoes = {
    "ICAO Empresa Aérea": "icao_empresa_aerea",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_autorizacao_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_aerodromo_origem",
    "ICAO Aeródromo Destino": "icao_aerodromo_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa"
}

bruto_padronizado = bruto.select(
    [
        F.col(col).alias(renomeacoes.get(col, col))
        for col in bruto.columns
    ]
)

In [0]:
for col in bruto_padronizado.columns:
    print(col)

In [0]:
bronze = bruto_padronizado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

In [0]:
bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA)

In [0]:
bronze.printSchema()

In [0]:
spark.sql(f"""
COMMENT ON TABLE {TABELA} IS
'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
Dado bruto: todas as colunas string, nenhuma linha descartada.
Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/.'
""")

In [0]:
display(
    spark.sql(f"""
        SELECT
            _arquivo_origem,
            COUNT(*) AS linhas,
            MAX (_ingerido_em) AS ingerido_em
        FROM {TABELA}
        GROUP BY _arquivo_origem
        ORDER BY _arquivo_origem
    """)
)

In [0]:
%sql
SHOW TABLES IN voebem.bronze